In [28]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("ResearchExamples").getOrCreate()

conf = SparkConf().setMaster('local[*]').setAppName("ResearchExamples")
sc = SparkContext.getOrCreate(conf = conf)

# 2. Create sample data and a DataFrame with a default partition count
data = [(i, f"User_{i}") for i in range(1, 31)]
df = spark.createDataFrame(data, ["id", "name"]).repartition(10)
print("\nPartitions before resizing:")
partitions = df.rdd.glom().collect()

# Print the rows separated by partition boundaries
for idx, partition in enumerate(partitions):
    print(f"--- Partition {idx} ---")
    print(f"Partition length: {len(partition)}")
    """
    for row in partition:
        print(row)
    """

# Repartition to decrease (Shuffles all data to ensure 2 equal-sized partitions)
df_repartition = df.repartition(3)
print("\nPartitions after repartition(3):") 
# Output: 2

# glom() gathers all elements within each partition into a list
partitions = df_repartition.rdd.glom().collect()

# Print the rows separated by partition boundaries
for idx, partition in enumerate(partitions):
    print(f"--- Partition {idx} ---")
    print(f"Partition length: {len(partition)}")
    """
    for row in partition:
        print(row)
    """

# Coalesce to decrease (Merges local partitions without a full shuffle)
df_coalesce = df.coalesce(3)
print("\nPartitions after coalesce(3):") 
# Output: 2

partitions = df_coalesce.rdd.glom().collect()

# Print the rows separated by partition boundaries
for idx, partition in enumerate(partitions):
    print(f"--- Partition {idx} ---")
    print(f"Partition length: {len(partition)}")
    """
    for row in partition:
        print(row)
    """

# Stop the Spark session
spark.stop()


Partitions before resizing:


--- Partition 0 ---
Partition length: 3
--- Partition 1 ---
Partition length: 5
--- Partition 2 ---
Partition length: 3
--- Partition 3 ---
Partition length: 3
--- Partition 4 ---
Partition length: 3
--- Partition 5 ---
Partition length: 3
--- Partition 6 ---
Partition length: 4
--- Partition 7 ---
Partition length: 1
--- Partition 8 ---
Partition length: 2
--- Partition 9 ---
Partition length: 3

Partitions after repartition(3):
--- Partition 0 ---
Partition length: 10
--- Partition 1 ---
Partition length: 10
--- Partition 2 ---
Partition length: 10

Partitions after coalesce(3):
--- Partition 0 ---
Partition length: 13
--- Partition 1 ---
Partition length: 9
--- Partition 2 ---
Partition length: 8
